# Dropout Uncertainty CoLoR Selection on Colab

This notebook runs the GPU-heavy MC-dropout scoring for `TASK_dropout_uncertainty_color_selection.md` and persists the raw `K` samples so local CPU analysis can try LCB/UCB, quantiles, triage, and later custom selection rules without rerunning inference.

Local code, pushed code, and runtime code are treated separately. Before running expensive cells, push the local changes that add `scripts/21_dropout_uncertainty_metrics.py`, `scripts/22_dropout_strategy_sweep.py`, and `configs/sweeps/score-parallel-dropout-uncertainty.yaml`, then replace `OLMO_SHA` below with that exact pushed commit SHA.

## 0. Resource Assumptions

Expected runtime:

- GPU: A100 preferred; L4/T4 may require smaller microbatch and longer wall time.
- GPU RAM: enough for one OLMo scorer at the selected microbatch.
- System RAM: high-RAM Colab recommended for wide raw-sample parquet writes.
- Local scratch: about 2 GB for configs/logs plus optional token copy.
- Drive quota: at least 5 GB free for raw score shards, MC sample artifacts, metrics, and logs for one K/dropout config.
- Remote downloads: none in this notebook if token pool, metadata, full scores, and checkpoints already exist on Drive.

The full run is sharded. Completed valid shards are skipped on rerun; partial or invalid production shards are not deleted automatically.

## 1. Runtime and Drive

One-time setup. Check GPU first.

In [ ]:
# PYTHON CELL
!nvidia-smi


In [ ]:
# PYTHON CELL
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# PYTHON CELL
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/color-filter-ablation')
SOURCE_ARTIFACT_ROOT = DRIVE / 'data' / 'pair_mid2_cascade_full_rerank'
DATA_DRIVE = SOURCE_ARTIFACT_ROOT / 'data'
MODELS_DRIVE = DRIVE / 'models'
SOURCE_RESULTS_DRIVE = SOURCE_ARTIFACT_ROOT / 'results'
RESULTS_DRIVE = DRIVE / 'results' / 'dropout-uncertainty'
RAW_SCORE_DRIVE = RESULTS_DRIVE / 'raw_score_shards'
REPORTS_DRIVE = DRIVE / 'reports' / 'dropout-uncertainty'
CONFIG_DRIVE = DRIVE / 'runtime_configs' / 'dropout-uncertainty'

for path in [MODELS_DRIVE, RESULTS_DRIVE, RAW_SCORE_DRIVE, REPORTS_DRIVE, CONFIG_DRIVE]:
    path.mkdir(parents=True, exist_ok=True)

print('drive root:', DRIVE)
print('source artifact root:', SOURCE_ARTIFACT_ROOT)
print('source data:', DATA_DRIVE)
print('models:', MODELS_DRIVE)
print('source results:', SOURCE_RESULTS_DRIVE)
print('dropout results:', RESULTS_DRIVE)
print('reports:', REPORTS_DRIVE)


In [ ]:
# PYTHON CELL
!df -h /content /content/drive/MyDrive


## 2. Clone, Pin, and Install

One-time setup. Replace `OLMO_SHA` with the pushed commit containing this notebook and scripts. The cell stops if the placeholder is still present.

In [ ]:
# PYTHON CELL
import subprocess
from pathlib import Path

OLMO_DIR = Path('/content/color-filter-olmo')
OLMO_REPO = 'https://github.com/myazdani/color-filter-olmo.git'
OLMO_SHA = 'REPLACE_WITH_PUSHED_COMMIT_SHA'

def run(*args, cwd=None):
    subprocess.run([str(arg) for arg in args], cwd=cwd, check=True)

def out(*args, cwd=None):
    return subprocess.check_output([str(arg) for arg in args], cwd=cwd, text=True).strip()

if OLMO_SHA == 'REPLACE_WITH_PUSHED_COMMIT_SHA':
    raise RuntimeError('Set OLMO_SHA to the pushed commit before running Colab GPU work.')

if not OLMO_DIR.exists():
    run('git', 'clone', OLMO_REPO, OLMO_DIR)
elif (OLMO_DIR / '.git').is_dir():
    run('git', '-C', OLMO_DIR, 'fetch', 'origin')
else:
    raise RuntimeError(f'{OLMO_DIR} exists but is not a git checkout')

run('git', '-C', OLMO_DIR, 'checkout', OLMO_SHA)
actual = out('git', '-C', OLMO_DIR, 'rev-parse', 'HEAD')
if actual != OLMO_SHA:
    raise RuntimeError(f'SHA mismatch: expected {OLMO_SHA}, got {actual}')
print('OLMo runtime SHA:', actual)


In [ ]:
# PYTHON CELL
import subprocess

overlay = [
    'omegaconf',
    'rich',
    'tokenizers',
    'transformers',
    'cached_path>=1.6.2',
    'packaging',
    'boto3',
    'google-cloud-storage',
    'wandb',
    'pandas',
    'pyarrow',
]
subprocess.run(['python', '-m', 'pip', 'install', '-q', *overlay], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', str(OLMO_DIR), '--no-deps'], check=True)
print('installed Colab overlay without reinstalling torch')


In [ ]:
# PYTHON CELL
import subprocess

required_paths = [
    OLMO_DIR / 'scripts/train.py',
    OLMO_DIR / 'scripts/21_dropout_uncertainty_metrics.py',
    OLMO_DIR / 'scripts/22_dropout_strategy_sweep.py',
    OLMO_DIR / 'configs/sweeps/score-parallel-dropout-uncertainty.yaml',
]
for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(path)
    print('ok:', path)

for script in ['21_dropout_uncertainty_metrics.py', '22_dropout_strategy_sweep.py']:
    subprocess.run(['python', '-m', 'py_compile', str(OLMO_DIR / 'scripts' / script)], check=True)
    subprocess.run(['python', str(OLMO_DIR / 'scripts' / script), '--help'], check=True, stdout=subprocess.PIPE)
print('script probes passed')


## 3. Configure Paths, Secrets, and Runtime-Local Config

Safe to rerun. Update these paths to match Drive. The production batch size defaults to 32 because 500,000 rows is divisible by 32 and the current OLMo scorer expects full batches.

In [ ]:
# PYTHON CELL
from pathlib import Path

COMMON_ARTIFACT_ROOTS = [
    SOURCE_ARTIFACT_ROOT,
    DRIVE,
    Path('/content/drive/MyDrive/color-filter-ablation/data/pair_mid2_cascade_full_rerank'),
    Path('/content/drive/MyDrive/CI-CoLoR/color-filter-ablation/data/pair_mid2_cascade_full_rerank'),
]

def first_existing(relative_paths, fallback):
    for root in COMMON_ARTIFACT_ROOTS:
        for relative in relative_paths:
            candidate = root / relative
            if candidate.exists():
                return candidate
    return fallback

TOKENS_DRIVE = first_existing(
    [Path('data/score_pool_tokens_official_500k.npy'), Path('score_pool_tokens_official_500k.npy')],
    DATA_DRIVE / 'score_pool_tokens_official_500k.npy',
)
META_DRIVE = first_existing(
    [Path('data/score_pool_meta_official_500k.parquet'), Path('score_pool_meta_official_500k.parquet')],
    DATA_DRIVE / 'score_pool_meta_official_500k.parquet',
)
FULL_SCORES_DRIVE = first_existing(
    [
        Path('results/score-pool-robustness-official-500k/scores_full.parquet'),
        Path('score-pool-robustness-official-500k/scores_full.parquet'),
        Path('scores_full.parquet'),
    ],
    SOURCE_RESULTS_DRIVE / 'score-pool-robustness-official-500k' / 'scores_full.parquet',
)

PRIOR_CHECKPOINT = first_existing(
    [
        Path('models/prior'),
        Path('assets/olmo/prior'),
        Path('assets/raw/prior'),
        Path('assets/prior'),
        Path('checkpoints/prior'),
        Path('prior'),
    ],
    MODELS_DRIVE / 'prior',
)
BOOKS_CHECKPOINT = first_existing(
    [
        Path('models/conditional_books'),
        Path('assets/olmo/conditional_books'),
        Path('assets/raw/conditional_books'),
        Path('assets/conditional_books'),
        Path('checkpoints/conditional_books'),
        Path('conditional_books'),
    ],
    MODELS_DRIVE / 'conditional_books',
)
HF_PRIOR_CHECKPOINT = first_existing(
    [Path('assets/hf/books_marg_hf'), Path('assets/hf/prior'), Path('assets/hf/books_marg')],
    DRIVE / 'assets' / 'hf' / 'books_marg_hf',
)
HF_BOOKS_CHECKPOINT = first_existing(
    [Path('assets/hf/books_cond_hf'), Path('assets/hf/conditional_books'), Path('assets/hf/books_cond')],
    DRIVE / 'assets' / 'hf' / 'books_cond_hf',
)

CONFIG_ID = 'dropout_k8_p005'
NUM_SAMPLES = 8
DROPOUT_RATE = 0.05
DROPOUT_TARGET = 'attention+residual+embedding'
SEED = 1
SEQ_LEN = 512
GLOBAL_BATCH_SIZE = 32
MICROBATCH = 32
SHARD_ROWS = 24_992
SMOKE_ROWS = 256
BENCH_ROWS = 1024
TAU64_CUTOFF = 0.3513622284

LOCAL_CONFIG_DIR = Path('/content/dropout_uncertainty_runtime_configs')
LOCAL_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

print('tokens:', TOKENS_DRIVE)
print('metadata:', META_DRIVE)
print('full scores:', FULL_SCORES_DRIVE)
print('prior checkpoint:', PRIOR_CHECKPOINT)
print('books checkpoint:', BOOKS_CHECKPOINT)
print('hf prior checkpoint candidate:', HF_PRIOR_CHECKPOINT, HF_PRIOR_CHECKPOINT.exists())
print('hf books checkpoint candidate:', HF_BOOKS_CHECKPOINT, HF_BOOKS_CHECKPOINT.exists())
print('config id:', CONFIG_ID)


## 4. Validate Input Artifacts

Safe to rerun. Stop if any shape, dtype, checkpoint, or row-count check fails.

In [ ]:
# PYTHON CELL
import numpy as np
import pandas as pd

required_paths = [TOKENS_DRIVE, META_DRIVE, FULL_SCORES_DRIVE, PRIOR_CHECKPOINT, BOOKS_CHECKPOINT]
missing_paths = [path for path in required_paths if not path.exists()]
for path in required_paths:
    print(('exists: ' if path.exists() else 'missing:'), path)
if missing_paths:
    raise FileNotFoundError(
        'Missing required Drive artifacts. The token/meta/full-score files are usually under '
        '/content/drive/MyDrive/color-filter-ablation/data/pair_mid2_cascade_full_rerank/. '
        'Check SOURCE_ARTIFACT_ROOT, MODELS_DRIVE, or copy the artifacts there. Missing: '
        + ', '.join(str(path) for path in missing_paths)
    )

for ckpt in [PRIOR_CHECKPOINT, BOOKS_CHECKPOINT]:
    expected = [ckpt / 'model.pt', ckpt / 'config.yaml']
    missing = [str(path) for path in expected if not path.exists()]
    if missing:
        hf_hint = ''
        if HF_PRIOR_CHECKPOINT.exists() or HF_BOOKS_CHECKPOINT.exists():
            hf_hint = (
                ' Found converted HF checkpoint candidates under assets/hf, but this notebook uses '
                'the native OLMo scorer and needs unsharded OLMo checkpoint directories containing '
                'model.pt and config.yaml. Either point PRIOR_CHECKPOINT/BOOKS_CHECKPOINT at those raw '
                'OLMo checkpoint dirs, or use a separate HF dropout scorer.'
            )
        raise FileNotFoundError(f'Missing expected unsharded checkpoint files: {missing}.{hf_hint}')
    print('checkpoint ok:', ckpt)

tokens = np.load(TOKENS_DRIVE, mmap_mode='r')
if tokens.ndim != 2 or tokens.shape[1] != SEQ_LEN:
    raise ValueError(f'Expected token shape [N,{SEQ_LEN}], got {tokens.shape}')
if tokens.dtype not in (np.uint16, np.uint32, np.int64):
    raise ValueError(f'Unexpected token dtype: {tokens.dtype}')
if tokens.shape[0] % GLOBAL_BATCH_SIZE != 0:
    raise ValueError(f'Token rows {tokens.shape[0]} not divisible by GLOBAL_BATCH_SIZE={GLOBAL_BATCH_SIZE}')

meta = pd.read_parquet(META_DRIVE)
full_scores = pd.read_parquet(FULL_SCORES_DRIVE)
if len(meta) < tokens.shape[0] or len(full_scores) < tokens.shape[0]:
    raise ValueError((len(meta), len(full_scores), tokens.shape[0]))
if 'pool_name' not in meta.columns:
    raise ValueError('metadata must contain pool_name for pairwise metrics')
print('tokens:', tokens.shape, tokens.dtype)
print('metadata rows:', len(meta), meta['pool_name'].value_counts().to_dict())
print('full score rows:', len(full_scores), 'columns:', list(full_scores.columns)[:12])


## 5. Convert or Load Models

The notebook expects unsharded OLMo checkpoints already on Drive. This section only prepares config helpers and validates one dry config; model loading happens in the bounded smoke gate.

In [ ]:
# PYTHON CELL
import math
import os
import subprocess
import sys
from pathlib import Path

from omegaconf import OmegaConf

sys.path.insert(0, str(OLMO_DIR))
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ.setdefault('WANDB_MODE', 'disabled')

TEMPLATE_CONFIG = OLMO_DIR / 'configs/sweeps/score-parallel-dropout-uncertainty.yaml'

def build_score_config(model_id, checkpoint_path, output_dir, rows, data_start_step=0, batch_size=GLOBAL_BATCH_SIZE, microbatch=MICROBATCH):
    if rows % batch_size != 0:
        raise ValueError(f'rows={rows} must be divisible by batch_size={batch_size}')
    cfg = OmegaConf.load(TEMPLATE_CONFIG)
    if 'sweep' in cfg:
        del cfg['sweep']
    cfg.run_name = f'{CONFIG_ID}_{model_id}_{data_start_step}'
    cfg.seed = SEED
    cfg.max_duration = rows // batch_size
    cfg.global_train_batch_size = batch_size
    cfg.device_train_microbatch_size = microbatch
    cfg.load_path = str(checkpoint_path)
    cfg.load_checkpoint_type = 'unsharded'
    cfg.save_folder = str(output_dir)
    cfg.data.paths = [str(TOKENS_DRIVE)]
    cfg.data.drop_last = True
    cfg.data.memmap_dtype = 'uint16' if tokens.dtype == np.uint16 else 'uint32'
    cfg.data_start_step = data_start_step
    cfg.uncertainty_scoring.enabled = True
    cfg.uncertainty_scoring.num_samples = NUM_SAMPLES
    cfg.uncertainty_scoring.perturbation_type = 'dropout'
    cfg.uncertainty_scoring.coupled_masks = True
    cfg.model.attention_dropout = DROPOUT_RATE
    cfg.model.residual_dropout = DROPOUT_RATE
    cfg.model.embedding_dropout = DROPOUT_RATE
    return cfg

def write_config(cfg, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    OmegaConf.save(cfg, path)
    print('wrote config:', path)

probe_cfg = build_score_config('prior_probe', PRIOR_CHECKPOINT, RESULTS_DRIVE / 'probe', GLOBAL_BATCH_SIZE)
probe_path = LOCAL_CONFIG_DIR / 'probe.yaml'
write_config(probe_cfg, probe_path)
print(OmegaConf.to_yaml(probe_cfg)[:1200])


## 6. Cheap Smoke Test / GPU Gate

Benchmark only / smoke only. This scores exactly `SMOKE_ROWS` rows for both prior and Books checkpoints into isolated `smoke/` outputs, then verifies raw MC samples and strategy sweeps. Do not proceed to production until this passes.

In [ ]:
# PYTHON CELL
import json
import subprocess
import time

def run_logged(args, log_path, cwd=OLMO_DIR, append=False):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    mode = 'a' if append else 'w'
    print('running:', ' '.join(str(a) for a in args))
    start = time.perf_counter()
    with log_path.open(mode) as log:
        proc = subprocess.run([str(a) for a in args], cwd=cwd, text=True, stdout=log, stderr=subprocess.STDOUT)
    elapsed = time.perf_counter() - start
    print('elapsed_seconds:', round(elapsed, 2), 'log:', log_path)
    if proc.returncode != 0:
        print(log_path.read_text(errors='ignore')[-4000:])
        raise RuntimeError(f'Command failed with exit code {proc.returncode}: {args}')

def infer_score_rows(score_dir):
    index_path = score_dir / 'mmap_index.npy'
    if not index_path.exists():
        return 0
    idx = np.memmap(index_path, dtype=np.int64, mode='r')
    end = len(idx)
    while end > 0:
        start = max(0, end - 1_000_000)
        chunk = np.asarray(idx[start:end])
        nz = np.flatnonzero(chunk)
        if len(nz):
            return start + int(nz[-1]) + 1
        end = start
    return 0

def score_width(score_dir):
    files = [line.strip() for line in (score_dir / 'files.txt').read_text().splitlines() if line.strip()]
    if not files:
        return 0
    path = Path(files[0])
    if not path.exists():
        path = score_dir / path.name
    values = path.stat().st_size // np.dtype(np.float32).itemsize
    return values // 1_048_576

def valid_score_dir(score_dir, expected_rows, expected_width=NUM_SAMPLES):
    if not (score_dir / 'files.txt').exists() or not (score_dir / 'mmap_index.npy').exists():
        return False
    return infer_score_rows(score_dir) == expected_rows and score_width(score_dir) >= expected_width

def run_score_once(model_id, checkpoint_path, output_dir, rows, data_start_step, config_name):
    score_dir = output_dir / 'score'
    if valid_score_dir(score_dir, rows):
        print('skipping valid score dir:', score_dir)
        return score_dir
    if output_dir.exists() and any(output_dir.iterdir()):
        raise RuntimeError(f'Output exists but is not a valid completed score dir: {output_dir}')
    cfg = build_score_config(model_id, checkpoint_path, output_dir, rows, data_start_step=data_start_step)
    cfg_path = LOCAL_CONFIG_DIR / f'{config_name}.yaml'
    write_config(cfg, cfg_path)
    log_path = output_dir.with_suffix('.log')
    run_logged([
        'torchrun', '--standalone', '--nproc_per_node=1', 'scripts/train.py', str(cfg_path), '--save_overwrite=true'
    ], log_path)
    if not valid_score_dir(score_dir, rows):
        raise RuntimeError(f'Score output failed validation: {score_dir}')
    return score_dir


In [ ]:
# PYTHON CELL
SMOKE_ROOT = RESULTS_DRIVE / 'smoke' / CONFIG_ID
smoke_prior = run_score_once('prior_smoke', PRIOR_CHECKPOINT, SMOKE_ROOT / 'prior', SMOKE_ROWS, 0, 'smoke_prior')
smoke_books = run_score_once('books_smoke', BOOKS_CHECKPOINT, SMOKE_ROOT / 'books', SMOKE_ROWS, 0, 'smoke_books')
print('smoke prior:', smoke_prior)
print('smoke books:', smoke_books)


In [ ]:
# PYTHON CELL
SMOKE_ANALYSIS = SMOKE_ROOT / 'analysis'
run_logged([
    'python', 'scripts/21_dropout_uncertainty_metrics.py',
    '--prior-score-dir', smoke_prior,
    '--conditional-score-dir', smoke_books,
    '--output-dir', SMOKE_ANALYSIS,
    '--config-id', f'{CONFIG_ID}_smoke',
    '--metadata', META_DRIVE,
    '--full-scores', FULL_SCORES_DRIVE,
    '--num-samples', NUM_SAMPLES,
    '--max-rows', SMOKE_ROWS,
    '--dropout-rate', DROPOUT_RATE,
    '--dropout-target', DROPOUT_TARGET,
    '--seed', SEED,
], SMOKE_ANALYSIS / 'aggregate.log')
run_logged([
    'python', 'scripts/22_dropout_strategy_sweep.py',
    '--mc-samples', SMOKE_ANALYSIS / f'mc_samples_{CONFIG_ID}_smoke.npz',
    '--summary', SMOKE_ANALYSIS / 'color_distribution_summary.parquet',
    '--output-dir', SMOKE_ANALYSIS / 'strategy',
    '--tau64-cutoff', TAU64_CUTOFF,
], SMOKE_ANALYSIS / 'strategy.log')

raw = np.load(SMOKE_ANALYSIS / f'mc_samples_{CONFIG_ID}_smoke.npz')
color_samples = raw['color_samples']
assert color_samples.shape == (SMOKE_ROWS, NUM_SAMPLES), color_samples.shape
assert np.isfinite(color_samples).all()
if np.allclose(color_samples[:, 0], color_samples[:, -1]):
    raise RuntimeError('First and last MC samples are identical; dropout may not be active')
print('smoke raw samples:', color_samples.shape, 'std mean:', color_samples.std(axis=1, ddof=1).mean())


## 7. Batch-Size and Shard-Size Tuning

Benchmark only. This is bounded to `BENCH_ROWS` rows and isolated under `benchmark/`. It tests the prior model only to choose a stable microbatch. If this fails with OOM, restart the runtime and use the largest smaller completed setting.

In [ ]:
# PYTHON CELL
BENCH_ROOT = RESULTS_DRIVE / 'benchmark' / CONFIG_ID
MICROBATCH_CANDIDATES = [16, 32, 64]
bench_results = []
for candidate in MICROBATCH_CANDIDATES:
    rows = BENCH_ROWS
    if rows % GLOBAL_BATCH_SIZE != 0:
        raise ValueError('BENCH_ROWS must be divisible by GLOBAL_BATCH_SIZE')
    out_dir = BENCH_ROOT / f'prior_microbatch_{candidate}'
    score_dir = out_dir / 'score'
    if valid_score_dir(score_dir, rows):
        print('benchmark already complete:', score_dir)
        bench_results.append((candidate, 'cached'))
        continue
    cfg = build_score_config('prior_bench', PRIOR_CHECKPOINT, out_dir, rows, data_start_step=0, microbatch=candidate)
    cfg_path = LOCAL_CONFIG_DIR / f'bench_prior_microbatch_{candidate}.yaml'
    write_config(cfg, cfg_path)
    try:
        run_logged([
            'torchrun', '--standalone', '--nproc_per_node=1', 'scripts/train.py', str(cfg_path), '--save_overwrite=true'
        ], out_dir.with_suffix('.log'))
        if not valid_score_dir(score_dir, rows):
            raise RuntimeError(f'Invalid benchmark score output: {score_dir}')
        bench_results.append((candidate, 'ok'))
    except Exception as exc:
        bench_results.append((candidate, f'failed: {exc}'))
        print('stopping benchmark ladder after failure')
        break
print('benchmark results:', bench_results)


## 8. Full Resumable Run

Full run. This scores both models in deterministic row shards. Safe to rerun: completed valid shard score directories are skipped. Invalid production shard directories stop the run for manual inspection.

In [ ]:
# PYTHON CELL
TOTAL_ROWS = int(tokens.shape[0])
if SHARD_ROWS % GLOBAL_BATCH_SIZE != 0:
    raise ValueError('SHARD_ROWS must be divisible by GLOBAL_BATCH_SIZE')

shards = []
start = 0
while start < TOTAL_ROWS:
    rows = min(SHARD_ROWS, TOTAL_ROWS - start)
    if rows % GLOBAL_BATCH_SIZE != 0:
        raise ValueError(f'Final shard rows not divisible by batch size: start={start}, rows={rows}')
    data_start_step = start // GLOBAL_BATCH_SIZE
    shards.append({'start': start, 'rows': rows, 'data_start_step': data_start_step})
    start += rows
print('total rows:', TOTAL_ROWS, 'num shards:', len(shards), 'first:', shards[:2], 'last:', shards[-1])


In [ ]:
# PYTHON CELL
PROD_ROOT = RAW_SCORE_DRIVE / CONFIG_ID

def run_model_shards(model_id, checkpoint_path):
    score_dirs = []
    for shard in shards:
        start = shard['start']
        rows = shard['rows']
        out_dir = PROD_ROOT / model_id / f'shard_{start:06d}_{start + rows:06d}'
        score_dir = run_score_once(
            model_id,
            checkpoint_path,
            out_dir,
            rows,
            shard['data_start_step'],
            f'{CONFIG_ID}_{model_id}_{start:06d}_{start + rows:06d}',
        )
        score_dirs.append(score_dir)
    return score_dirs

prior_score_dirs = run_model_shards('prior', PRIOR_CHECKPOINT)
books_score_dirs = run_model_shards('books', BOOKS_CHECKPOINT)
print('prior shards:', len(prior_score_dirs))
print('books shards:', len(books_score_dirs))


## 9. Resume After Disconnect

Safe to rerun. After reconnect, rerun Sections 1, 2, 3, 4, 5, and the shard-definition cell in Section 8. Then run this status cell and rerun the full model-shard cell; it skips valid completed shards.

In [ ]:
# PYTHON CELL
def shard_status(model_id):
    rows = []
    for shard in shards:
        start = shard['start']
        expected = shard['rows']
        score_dir = PROD_ROOT / model_id / f'shard_{start:06d}_{start + expected:06d}' / 'score'
        rows.append({
            'model': model_id,
            'start': start,
            'rows': expected,
            'exists': score_dir.exists(),
            'valid': valid_score_dir(score_dir, expected) if score_dir.exists() else False,
            'score_dir': str(score_dir),
        })
    return rows

status = shard_status('prior') + shard_status('books')
complete = sum(1 for row in status if row['valid'])
print('valid shard score dirs:', complete, '/', len(status))
for row in status[:5] + status[-5:]:
    print(row)
missing = [row for row in status if not row['valid']]
print('missing/invalid count:', len(missing))
if missing:
    print('first missing/invalid:', missing[0])


## 10. Metrics, Report, and Outputs to Bring Back

Safe to rerun after all score shards are complete. This writes raw MC sample artifacts, summary parquet, strategy metrics, selected-index files, and logs directly under Drive.

In [ ]:
# PYTHON CELL
ANALYSIS_DIR = RESULTS_DRIVE / CONFIG_ID / 'analysis'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

prior_dirs = [str(PROD_ROOT / 'prior' / f"shard_{s['start']:06d}_{s['start'] + s['rows']:06d}" / 'score') for s in shards]
books_dirs = [str(PROD_ROOT / 'books' / f"shard_{s['start']:06d}_{s['start'] + s['rows']:06d}" / 'score') for s in shards]
for path in prior_dirs + books_dirs:
    if not Path(path).exists():
        raise FileNotFoundError(path)

aggregate_cmd = [
    'python', 'scripts/21_dropout_uncertainty_metrics.py',
    '--prior-score-dir', *prior_dirs,
    '--conditional-score-dir', *books_dirs,
    '--output-dir', ANALYSIS_DIR,
    '--config-id', CONFIG_ID,
    '--metadata', META_DRIVE,
    '--full-scores', FULL_SCORES_DRIVE,
    '--num-samples', NUM_SAMPLES,
    '--dropout-rate', DROPOUT_RATE,
    '--dropout-target', DROPOUT_TARGET,
    '--seed', SEED,
]
run_logged(aggregate_cmd, ANALYSIS_DIR / 'aggregate.log')

strategy_cmd = [
    'python', 'scripts/22_dropout_strategy_sweep.py',
    '--mc-samples', ANALYSIS_DIR / f'mc_samples_{CONFIG_ID}.npz',
    '--summary', ANALYSIS_DIR / 'color_distribution_summary.parquet',
    '--output-dir', ANALYSIS_DIR / 'strategy',
    '--tau64-cutoff', TAU64_CUTOFF,
]
run_logged(strategy_cmd, ANALYSIS_DIR / 'strategy.log')
print('analysis dir:', ANALYSIS_DIR)


In [ ]:
# PYTHON CELL
expected_outputs = [
    ANALYSIS_DIR / f'mc_samples_{CONFIG_ID}.npz',
    ANALYSIS_DIR / f'mc_samples_{CONFIG_ID}.parquet',
    ANALYSIS_DIR / 'color_distribution_summary.parquet',
    ANALYSIS_DIR / f'mc_samples_{CONFIG_ID}_manifest.json',
    ANALYSIS_DIR / 'strategy' / 'strategy_sweep_metrics.csv',
    ANALYSIS_DIR / 'strategy' / 'strategy_selection_overlap.csv',
]
for path in expected_outputs:
    if not path.exists():
        raise FileNotFoundError(path)
    print('output:', path, 'size_mb=', round(path.stat().st_size / 1e6, 2))

summary = pd.read_parquet(ANALYSIS_DIR / 'color_distribution_summary.parquet')
metrics = pd.read_csv(ANALYSIS_DIR / 'strategy' / 'strategy_sweep_metrics.csv')
print('summary rows:', len(summary), 'columns:', list(summary.columns)[:20])
print('metric rows:', len(metrics), 'scopes:', metrics['metric_scope'].value_counts().to_dict())
print(metrics.head())


## 11. Output Review and Acceptance Checks

Review these before interpreting results. Completion requires raw MC samples, summary statistics, strategy metrics, and selected-index artifacts on Drive, not only in `/content`.

In [ ]:
# PYTHON CELL
raw = np.load(ANALYSIS_DIR / f'mc_samples_{CONFIG_ID}.npz')
color_samples = raw['color_samples']
assert color_samples.shape == (TOTAL_ROWS, NUM_SAMPLES), color_samples.shape
assert np.isfinite(color_samples).all()
assert len(summary) == TOTAL_ROWS
assert len(metrics) > 0
selected_dir = ANALYSIS_DIR / 'strategy' / 'strategy_selected_indices'
selected_files = sorted(selected_dir.glob('*.npy'))
if not selected_files:
    raise FileNotFoundError(selected_dir)
print('raw sample shape:', color_samples.shape)
print('mean per-row MC std:', float(color_samples.std(axis=1, ddof=1).mean()))
print('selected-index files:', len(selected_files))
print('Drive analysis complete:', ANALYSIS_DIR)
